<a href="https://colab.research.google.com/github/eduhuemar001/llm-finetuning/blob/main/sentiment-data-classify.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Libraries

In [4]:
!pip install transformers torch huggingface_hub datasets pandas
import re
import json
from tqdm import tqdm
from transformers import pipeline
from google.colab import files
from huggingface_hub import login
from datasets import load_dataset

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 12.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.


In [16]:
login()

### Preprocessing

In [10]:
uploaded = files.upload()

comments = []
with open("youtube_comments_clean.txt", "r", encoding="utf-8") as f:
    for line in f:
        text = line.strip()
        if text:
            comments.append(text)

comments = comments[:10000]
print(f"Loaded {len(comments)} comments")
print("Sample:", comments[0])

Saving youtube_comments_clean.txt to youtube_comments_clean (2).txt
Loaded 10000 comments
Sample: Ich liebe die neuen „Idee-Losen“ Spiele … Grüsse gehen raus an das PP-Team


### BERT pipeline




In [11]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TextClassificationPipeline

def load_pipeline(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to("cuda")
    return TextClassificationPipeline(model=model, tokenizer=tokenizer, device=0)

model1 = pipeline("sentiment-analysis",
                  model="oliverguhr/german-sentiment-bert",
                  tokenizer="oliverguhr/german-sentiment-bert",
                  device=0, truncation=True)

model2 = pipeline("sentiment-analysis",
                  model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
                  tokenizer="cardiffnlp/twitter-xlm-roberta-base-sentiment",
                  device=0, truncation=True)

model3 = pipeline("sentiment-analysis",
                  model="nlptown/bert-base-multilingual-uncased-sentiment",
                  tokenizer="nlptown/bert-base-multilingual-uncased-sentiment",
                  device=0, truncation=True)

Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0


In [12]:
def normalize(label):
    label = label.lower()
    if "neg" in label:
        return "negative"
    elif "pos" in label:
        return "positive"
    elif "neu" in label:
        return "neutral"
    return "neutral"

def majority_vote(preds1, preds2, preds3):
    final_labels = []
    for a, b, c in zip(preds1, preds2, preds3):
        votes = [
            normalize(a["label"]),
            normalize(b["label"]),
            normalize(c["label"])
        ]
        majority = max(set(votes), key=votes.count)
        final_labels.append(majority if votes.count(majority) >= 2 else "uncertain")
    return final_labels

# === Batched classification ===
batch_size = 32
output_path = "youtube_comments_labeled.jsonl"

with open(output_path, "w", encoding="utf-8") as out_file:
    for i in tqdm(range(0, len(comments), batch_size), desc="Labeling"):
        batch = [c[:1000] for c in comments[i:i + batch_size]]  # safely truncate text

        preds1 = model1(batch)
        preds2 = model2(batch)
        preds3 = model3(batch)

        labels = majority_vote(preds1, preds2, preds3)

        for text, label in zip(batch, labels):
            json.dump({"text": text, "label": label}, out_file, ensure_ascii=False)
            out_file.write("\n")

Labeling: 100%|██████████| 313/313 [04:52<00:00,  1.07it/s]


In [13]:
files.download(output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
sb10k = load_dataset("Alienmaster/SB10k", split="train")
sb10k.to_csv("sb10k.tsv", sep="\t", index=False)
files.download("sb10k.tsv")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

train.tsv:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

test.tsv:   0%|          | 0.00/683k [00:00<?, ?B/s]

dev.tsv:   0%|          | 0.00/348k [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/1496 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/747 [00:00<?, ? examples/s]

Creating CSV from Arrow format:   0%|          | 0/6 [00:00<?, ?ba/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>